# Alternative ion forms: co-elution and exact-mass-offset screen (Supplementary Figures 6 and 7)

**Run order**
1. *Get the MassQL results to retrieve the RT*: download the MassQL result tables of all queries (5 repository collections x 25 queries) from GNPS2 and harmonise the core names.
2. *Loop*: for every cluster retained in the final libraries, trace back all contributing scans (pre-Falcon annotation tables + Falcon cluster CSVs) and attach their retention time.
3. Run `ISF_C27_tetra.ipynb`: it regenerates the C27 tetrahydroxy table, whose RTs were not all recovered by the loop.
4. *Determine what is a potential artifact*: QC summary of the per-core tables and `pepmass` → `precmz` rename.
5. *Per core*: `artifact_screen.screen_artifacts` flags pairs within each file (20 ppm, 10 s) and draws the Sankey diagrams of Supplementary Figure 6.
6. *Cores combined*: the same per library (Supplementary Figure 7).
7. *Update the compound names in the libraries*: write the ion-form flags into the final library annotation tables.

`artifact_screen.py` must sit next to this notebook.

In [ ]:
import os

# Root of the local project folder (the one holding Falcon/, Queries_final_mgf/,
# final_mgfs_library_generation/, ...). Edit this single line to run elsewhere.
PROJECT_DIR = '/media/dorrestein/Helena2/Bile_acids_update'

In [ ]:
import pandas as pd

## Get the MassQL results to retrieve the RT

In [ ]:
task_massql_results_C23_mono = {
    'metabolights': 'https://gnps2.org/status?task=5ca7383070824a379ee23b1d05ce52eb',
    'workbench': 'https://gnps2.org/status?task=46dc0dede70b4fe08863de330f7c556d',
    'massive_orbitrap': 'https://gnps2.org/status?task=7aea3a540340420da499ad00f907acb2',
    'massive_qe': 'https://gnps2.org/status?task=b3fa39575c18480c8292f4d415305608',
    'massive_qtof': 'https://gnps2.org/status?task=20342add5b904d379dabcb1f613f41cd'
}

task_massql_results_C23_di = {
    'metabolights': 'https://gnps2.org/status?task=c40aca5021f042dc878b7da938550c1b',
    'workbench': 'https://gnps2.org/status?task=24f2bce574cd49e3a6d4c9fc932396a0',
    'massive_orbitrap': 'https://gnps2.org/status?task=f1c37054c71b4e57b3335d82282aa3c6',
    'massive_qe': 'https://gnps2.org/status?task=4fc6f6bbacff4a7ebd5b89292ef72ba4',
    'massive_qtof': 'https://gnps2.org/status?task=29ea3333cdf44653bb13c0a5752c4caf'
}

task_massql_results_C23_tri = {
    'metabolights': 'https://gnps2.org/status?task=94e297fd9ec649759289d126f34513e4',
    'workbench': 'https://gnps2.org/status?task=f516ee61a6d04f048329dbbf16a5e34d',
    'massive_orbitrap': 'https://gnps2.org/status?task=43c7df7b684442e0a1400d17fb217c6c',
    'massive_qe': 'https://gnps2.org/status?task=62d5c228bd2b40e2829b790eb69fc872',
    'massive_qtof': 'https://gnps2.org/status?task=ac4d3b9a09184ee1a7652670e800f480'
}

task_massql_results_C23_tetra = {
    'metabolights': 'https://gnps2.org/status?task=e0b436ed61464f22b389b71d366669de',
    'workbench': 'https://gnps2.org/status?task=4326b94cf4d547909216b741f108f154',
    'massive_orbitrap': 'https://gnps2.org/status?task=91645ebbb07b4a92a8c374b0a4eb2fb6',
    'massive_qe': 'https://gnps2.org/status?task=44c67e218c594d24a3b10ee587ad4d0d',
    'massive_qtof': 'https://gnps2.org/status?task=e07f9be1a7d442a1829b9ff54e6fcaf5'
}

task_massql_results_C23_penta = {
    'metabolights': 'https://gnps2.org/status?task=d2d40b2fcf8248b5a12e95ec103dc48a',
    'workbench': 'https://gnps2.org/status?task=5d0944647987468c9417755e2cbfd747',
    'massive_orbitrap': 'https://gnps2.org/status?task=655ee07d6ae54515972d919b95c36d25',
    'massive_qe': 'https://gnps2.org/status?task=65c1a191067748f8bfa0cef285f24d4d',
    'massive_qtof': 'https://gnps2.org/status?task=c115999092594b1387bcd5ef1521b5a1'
}

task_massql_results_C24_non = {
    'metabolights': 'https://gnps2.org/status?task=1876bb8224ad4f4cb6740b58f1190710',
    'workbench': 'https://gnps2.org/status?task=846770dca42e4959a092c77bf4a81e70',
    'massive_orbitrap': 'https://gnps2.org/status?task=6f2fb946a6ad4da9afe3d3b45f1eda74',
    'massive_qe': 'https://gnps2.org/status?task=a145ea8a170541ea99a28e3daeca3df3',
    'massive_qtof': 'https://gnps2.org/status?task=e8d2b78c485943559e8ae2755b584475'
}

task_massql_results_C24_mono = {
    'metabolights': 'https://gnps2.org/status?task=38dd434754e24d5da59c12cc8d400eb0 ',
    'workbench': 'https://gnps2.org/status?task=f47c183a9877430a95ced5343fa04526',
    'massive_orbitrap': 'https://gnps2.org/status?task=ee126b61a6834f088d155e1443efdbe1',
    'massive_qe': 'https://gnps2.org/status?task=19fcb29a73a344e9bdf02db664bb8f99',
    'massive_qtof': 'https://gnps2.org/status?task=23e1db0f0c234f73afa180a416b349b9'
}

task_massql_results_C24_di = {
    'metabolights': 'https://gnps2.org/status?task=9afa0e7389ec4274b9393e5b948f52d3',
    'workbench': 'https://gnps2.org/status?task=75342b868e3d4631aa3199191d4b34d1',
    'massive_orbitrap': 'https://gnps2.org/status?task=7eddd658dd3147b897f5e08792f63fc8',
    'massive_qe': 'https://gnps2.org/status?task=40b84c51b0894acab21a7a49f289fd17',
    'massive_qtof': 'https://gnps2.org/status?task=fa0bc3d972574a00b0951f042aa1bfef'
}

task_massql_results_C24_tri = {
    'metabolights': 'https://gnps2.org/status?task=979c506c312c41cbad12a586e3067169',
    'workbench': 'https://gnps2.org/status?task=6ab1cfedd0784dd982a171d739f8b3aa',
    'massive_orbitrap': 'https://gnps2.org/status?task=f38295d12dbb404a9dcb4f69c31f0f97',
    'massive_qe': 'https://gnps2.org/status?task=c4bff9922d0a4935964c05b939299964',
    'massive_qtof': 'https://gnps2.org/status?task=8454249d32f9455993d49df50236145c'
}

task_massql_results_C24_tetra = {
    'metabolights': 'https://gnps2.org/status?task=765dc4e63f9c48dab831f6a5ea42a03b',
    'workbench': 'https://gnps2.org/status?task=069bcfa86a24479c9879ff70194750c7',
    'massive_orbitrap': 'https://gnps2.org/status?task=5436c2f57f894c1786151a15fb0bc99a',
    'massive_qe': 'https://gnps2.org/status?task=a4d3a9cb9a9f4098bf7b8bcc87e8107e',
    'massive_qtof': 'https://gnps2.org/status?task=dee1485c4fa245188fee533f1fbe5496'
}

task_massql_results_C24_penta = {
    'metabolights': 'https://gnps2.org/status?task=a02f387eb432455e99ffad327da49f3c',
    'workbench': 'https://gnps2.org/status?task=9157bb7d5f154ec2b07b7d750e8da049',
    'massive_orbitrap': 'https://gnps2.org/status?task=76f94524f32640048c8784b2b9d8374f',
    'massive_qe': 'https://gnps2.org/status?task=3f8172383b6b4381a94b59cbc30678d7',
    'massive_qtof': 'https://gnps2.org/status?task=301c5f64728d4e79be31930d3a1cce46'
}

task_massql_results_C24_alkamine_mono = {
    'metabolights': 'https://gnps2.org/status?task=fa0e3381ae65400da290329ef30d1408',
    'workbench': 'https://gnps2.org/status?task=b0e73aa7d4d04de78a9b13c27d1b3e92',
    'massive_orbitrap': 'https://gnps2.org/status?task=d3ec6c4ab67448e2bf19fb498b98cee1',
    'massive_qe': 'https://gnps2.org/status?task=2312761a97364aeab211032e2aeed3f3',
    'massive_qtof': 'https://gnps2.org/status?task=9d673d01289d4659a506e39348c2e284'
}

task_massql_results_C24_alkamine_di = {
    'metabolights': 'https://gnps2.org/status?task=c18d1f2459814b5c8b2cdc9128e0252e',
    'workbench': 'https://gnps2.org/status?task=9045d537fd544a03bf2e68ad5f8666d6',
    'massive_orbitrap': 'https://gnps2.org/status?task=cec67da823c74f8ba88e18507716d3c1',
    'massive_qe': 'https://gnps2.org/status?task=62de23e4a64b43ffbe3a74ae6d08c022',
    'massive_qtof': 'https://gnps2.org/status?task=a1780a0630d04ce09ed6f459b668b267'
}

task_massql_results_C24_alkamine_tri = {
    'metabolights': 'https://gnps2.org/status?task=84b9bbcc63904b26898ea4a6ebcf5ac5',
    'workbench': 'https://gnps2.org/status?task=b63887fc9a5a4c1faf07c07e6b1af725',
    'massive_orbitrap': 'https://gnps2.org/status?task=8233798235fe409292ea9963fa8dc2eb',
    'massive_qe': 'https://gnps2.org/status?task=4e4f596995a341818b7add7e1805524a',
    'massive_qtof': 'https://gnps2.org/status?task=a6d88aa98c7a43c1bb0198ada4f0deaa'
}

task_massql_results_C24_alkamine_tetra = {
    'metabolights': 'https://gnps2.org/status?task=b655a186dbec401cba15feb9576b47cc',
    'workbench': 'https://gnps2.org/status?task=8399128026524f2c90d2a3c2e81893c2',
    'massive_orbitrap': 'https://gnps2.org/status?task=a389c5e88d464664812f6a728cc58b5d',
    'massive_qe': 'https://gnps2.org/status?task=6339d292bfdd4165b69d52881a6ccc58',
    'massive_qtof': 'https://gnps2.org/status?task=93f81902f4e1431b90949507358f5dcf'
}

task_massql_results_C24_alkamine_penta = {
    'metabolights': 'https://gnps2.org/status?task=2fbd4f13c6864742952c6a26e6e02872',
    'workbench': 'https://gnps2.org/status?task=8593417597224b51ab667c1f8d3b1af2',
    'massive_orbitrap': 'https://gnps2.org/status?task=f2c167b0fa354b4bb5bcce4764f948fd',
    'massive_qe': 'https://gnps2.org/status?task=f4e2c538631f49e29595d5fe3c8ffa10',
    'massive_qtof': 'https://gnps2.org/status?task=38a7fc3f2a5f421487810962d5be6225'
}

task_massql_results_C26_di = {
    'metabolights': 'https://gnps2.org/status?task=86fb2a94a0f64b83a4fbfde91ea75587',
    'workbench': 'https://gnps2.org/status?task=0c47f87da0694ffd8f8d9002bdd7f2f6',
    'massive_orbitrap': 'https://gnps2.org/status?task=d835299a44a8444d9bc51d7a83cf33a4',
    'massive_qe': 'https://gnps2.org/status?task=4f45f9c6bfb94f42bc8731eecf315362',
    'massive_qtof': 'https://gnps2.org/status?task=f514cb7497bc485ea18c23b8725f9ba5'
}

task_massql_results_C27_mono = {
    'metabolights': 'https://gnps2.org/status?task=23a0f9594d6748fb96579e47897945f1',
    'workbench': 'https://gnps2.org/status?task=ca9f6a3caa774317a93ed1eaccdbb0eb',
    'massive_orbitrap': 'https://gnps2.org/status?task=b3534050d39946738a0ddfdfae9a2378',
    'massive_qe': 'https://gnps2.org/status?task=b25fbb415830486ea3c6f701ce364b32',
    'massive_qtof': 'https://gnps2.org/status?task=fdaf696c43d74d4c8ec329990e290a23'
}

task_massql_results_C27_di = {
    'metabolights': 'https://gnps2.org/status?task=a29fda083f4747fca53e1801e5ad4f53',
    'workbench': 'https://gnps2.org/status?task=64a75dcab0fe4567ae1bb8ca77e711d0',
    'massive_orbitrap': 'https://gnps2.org/status?task=1cf3aadc03f94154b2211625e07471ef',
    'massive_qe': 'https://gnps2.org/status?task=e6ca328938494013975afb8a64d51151',
    'massive_qtof': 'https://gnps2.org/status?task=74fcacc2c93d444ea5c2fef3fb43957f'
}

task_massql_results_C27_tri = {
    'metabolights': 'https://gnps2.org/status?task=8cdc48ee33894afab184b9cd2c5cc6ec',
    'workbench': 'https://gnps2.org/status?task=e46559fed15a4b71821fab007786c15b',
    'massive_orbitrap': 'https://gnps2.org/status?task=071aa9038a7347e8a18593b029352e26',
    'massive_qe': 'https://gnps2.org/status?task=5ad8796ba3ef4dcf89fab3fd1e4da188',
    'massive_qtof': 'https://gnps2.org/status?task=921d4609e6304d54b64f68bde7ce351b'
}

task_massql_results_C27_tetra = {
    'metabolights': 'https://gnps2.org/status?task=9248e455b74a4f039092bb8e5f275a97',
    'workbench': 'https://gnps2.org/status?task=39c43758dbb749979c8830b925438455',
    'massive_orbitrap': 'https://gnps2.org/status?task=20cb981fadbf4e7fae0294fe6c9223c4',
    'massive_qe': 'https://gnps2.org/status?task=3c63e00d8965465fa6a23166042fc552',
    'massive_qtof': 'https://gnps2.org/status?task=8b0fc03923c24a72ab0c081fc0fbe37d'
}

task_massql_results_C27_penta = {
    'metabolights': 'https://gnps2.org/status?task=e9d2facda805496fa2f968ffb27804f6',
    'workbench': 'https://gnps2.org/status?task=41f89b8989664d399f368a989e0ce057',
    'massive_orbitrap': 'https://gnps2.org/status?task=3642a549ab3848a6967943bb49006b98',
    'massive_qe': 'https://gnps2.org/status?task=f4504d7708e343a18967f9115ff9c3c3',
    'massive_qtof': 'https://gnps2.org/status?task=d144c5e408c5410f97b8e631650d55fb'
}

task_massql_results_C28_alcohol_1OH = {
    'metabolights': 'https://gnps2.org/status?task=02a3ee4d94ad48f59a76e81491baf06c',
    'workbench': 'https://gnps2.org/status?task=4a1188efe0264eb59a084dd391573c66',
    'massive_orbitrap': 'https://gnps2.org/status?task=ea4ae131d2734da08e914d4e0cf25930',
    'massive_qe': 'https://gnps2.org/status?task=a3d3d0e5578742dba348414652007631',
    'massive_qtof': 'https://gnps2.org/status?task=271bb5abf55d42d9b902568bddb1c175'
}

task_massql_results_C28_alcohol_2OH = {
    'metabolights': 'https://gnps2.org/status?task=2571dc2107f64147af4949b5305b645a',
    'workbench': 'https://gnps2.org/status?task=8af135a4ba674dc1bf92f77aa5db3d80',
    'massive_orbitrap': 'https://gnps2.org/status?task=0aa5b1b44aae40e8af354a784a913977',
    'massive_qe': 'https://gnps2.org/status?task=9e4a38440fff4876a720015724eec019',
    'massive_qtof': 'https://gnps2.org/status?task=b2f2d2989d1b4807ac61290be4f49e0a'
}

task_massql_results_C28_alcohol_3OH = {
    'metabolights': 'https://gnps2.org/status?task=db19dee5978445c89ea8c29f4e2d4294',
    'workbench': 'https://gnps2.org/status?task=de7e6b00f5704310a274659682f2045d',
    'massive_orbitrap': 'https://gnps2.org/status?task=0b44e69f29064e468adbe9b679df72a1',
    'massive_qe': 'https://gnps2.org/status?task=93eaf4be688646619f168635d32deada',
    'massive_qtof': 'https://gnps2.org/status?task=e37b808a6ce34b53bced843725928b88'
}

In [ ]:
import io
import time
import requests
import pandas as pd

# Grab every dict named task_massql_results_* from the current namespace
all_tasks = {
    name: obj
    for name, obj in list(globals().items())
    if name.startswith('task_massql_results_') and isinstance(obj, dict)
}

# Total count for progress tracking
total = sum(len(d) for d in all_tasks.values())

frames = []
counter = 0
for ba_core, repo_dict in all_tasks.items():
    for repository, status_url in repo_dict.items():
        counter += 1
        task_id = status_url.split('task=')[1].strip()
        url = f'https://gnps2.org/resultfile?task={task_id}&file=nf_output/extracted/extracted.tsv'
        try:
            response = requests.get(url)
            response.raise_for_status()
            df = pd.read_csv(io.StringIO(response.text), sep='\t')
            df['BA_core'] = ba_core
            df['repository'] = repository
            frames.append(df)
            print(f'[{counter}/{total}] OK      {ba_core} / {repository} -> {len(df)} rows')
        except Exception as e:
            print(f'[{counter}/{total}] FAILED  {ba_core} / {repository} ({task_id}): {e}')

        time.sleep(5)   # pause between requests

combined = pd.concat(frames, ignore_index=True)
print(f'\nDone. Combined {len(frames)} files into {len(combined)} total rows.')

In [ ]:
import io
import requests
import pandas as pd

def fetch_one(ba_core, repository, task_id):
    task_id = task_id.strip()
    url = f'https://gnps2.org/resultfile?task={task_id}&file=nf_output/extracted/extracted.tsv'
    response = requests.get(url)
    response.raise_for_status()
    df = pd.read_csv(io.StringIO(response.text), sep='\t')
    df['BA_core'] = ba_core
    df['repository'] = repository
    print(f'OK  {ba_core} / {repository} -> {len(df)} rows')
    return df

# Clear any prior attempt for this task so reruns don't duplicate rows
mask = (combined['BA_core'] == 'task_massql_results_C26_di') & (combined['repository'] == 'massive_qtof')
combined = combined[~mask]

retry_df = fetch_one('task_massql_results_C26_di', 'massive_qtof', 'f514cb7497bc485ea18c23b8725f9ba5')

combined = pd.concat([combined, retry_df], ignore_index=True)
print(f'Combined now has {len(combined)} total rows.')

In [ ]:
combined.to_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/combined_all_massql_results.tsv', sep='\t', index=False)

In [ ]:
import polars as pl

combined = pl.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/combined_all_massql_results.tsv', separator='\t')

# Strip the prefix
combined = combined.with_columns(
    pl.col('BA_core').str.replace('task_massql_results_', '')
)

mapping = {
    'C23_mono': 'C23_monohydroxy',
    'C23_di': 'C23_dihydroxy',
    'C23_tri': 'C23_trihydroxy',
    'C23_tetra': 'C23_tetrahydroxy',
    'C23_penta': 'C23_pentahydroxy',

    'C24_non': 'C24_nonhydroxy',
    'C24_mono': 'C24_monohydroxy',
    'C24_di': 'C24_dihydroxy',
    'C24_tri': 'C24_trihydroxy',
    'C24_tetra': 'C24_tetrahydroxy',
    'C24_penta': 'C24_pentahydroxy',

    'C24_alkamine_mono': 'C24_alkylamine_monohydroxy',
    'C24_alkamine_di': 'C24_alkylamine_dihydroxy',
    'C24_alkamine_tri': 'C24_alkylamine_trihydroxy',
    'C24_alkamine_tetra': 'C24_alkylamine_tetrahydroxy',
    'C24_alkamine_penta': 'C24_alkylamine_pentahydroxy',

    'C26_di': 'Aceto_dihydroxy',

    'C27_mono': 'C27_monohydroxy',
    'C27_di': 'C27_dihydroxy',
    'C27_tri': 'C27_trihydroxy',
    'C27_tetra': 'C27_tetrahydroxy',
    'C27_penta': 'C27_pentahydroxy',
    # C28 alcohols unchanged — no need to list them
}

combined = combined.with_columns(
    pl.col('BA_core').replace(mapping)
)

In [ ]:
combined.write_csv(f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/combined_all_massql_results_cleaned.tsv', separator='\t')

## Merge RT with falcon results for final library generated

In [ ]:
pre_falcon = {
    'C24_nonhydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Nonhydroxy/download_mgfs/final_mgf/C24_Nonhydroxy_input_falcon_charge_annotation_table.tsv',
    'C24_monohydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Monohydroxy/download_mgfs/final_mgf/C24_Monohydroxy_input_falcon_charge_annotation_table.tsv',
    'C24_dihydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Dihydroxy/download_mgfs/final_mgf/C24_dihydroxy_input_falcon_charge_annotation_table.tsv',
    'C24_trihydroxy_part1': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/download_mgfs/final_mgf/C24_Trihydroxy_input_falcon_charge_part1_annotation_table.tsv',
    'C24_trihydroxy_part2': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/download_mgfs/final_mgf/C24_Trihydroxy_input_falcon_charge_part2_annotation_table.tsv',
    'C24_trihydroxy_part3': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/download_mgfs/final_mgf/C24_Trihydroxy_input_falcon_charge_part3_annotation_table.tsv',
    'C24_tetrahydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Tetrahydroxy/download_mgfs/final_mgf/C24_Tetrahydroxy_input_falcon_charge_annotation_table.tsv',
    'C24_pentahydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Pentahydroxy/download_mgfs/final_mgf/C24_Pentahydroxy_input_falcon_charge_annotation_table.tsv',

    "C23_monohydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_monohydroxy/download_mgfs/final_mgf/C23_Monohydroxy_input_falcon_charge_annotation_table.tsv',
    "C23_dihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_dihydroxy/download_mgfs/final_mgf/C23_dihydroxy_input_falcon_charge_annotation_table.tsv',
    "C23_trihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_trihydroxy/download_mgfs/final_mgf/C23_trihydroxy_input_falcon_charge_annotation_table.tsv',
    "C23_tetrahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_tetrahydroxy/download_mgfs/final_mgf/C23_tetrahydroxy_input_falcon_charge_annotation_table.tsv',
    "C23_pentahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_pentahydroxy/download_mgfs/final_mgf/C23_pentahydroxy_input_falcon_charge_annotation_table.tsv',

    "C27_monohydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_monohydroxy/download_mgfs/final_mgf/C27_monohydroxy_input_falcon_charge_annotation_table.tsv',
    "C27_dihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_dihydroxy/download_mgfs/final_mgf/C27_dihydroxy_input_falcon_charge_annotation_table.tsv',
    "C27_trihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_trihydroxy/download_mgfs/final_mgf/C27_trihydroxy_input_falcon_charge_annotation_table.tsv',
    "C27_tetrahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_tetrahydroxy/download_mgfs/final_mgf/C27_tetrahydroxy_input_falcon_charge_annotation_table.tsv',
    "C27_pentahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_pentahydroxy/download_mgfs/final_mgf/C27_pentahydroxy_input_falcon_charge_annotation_table.tsv',

    "Aceto_dihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/Aceto/Dihydroxy/download_mgfs/final_mgf/Aceto_dihydroxy_input_falcon_charge_renumbered_annotation_table.tsv',

    "C24_alkylamine_monohydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_monohydroxy/download_mgfs/final_mgf/C24_alkylamine_monohydroxy_input_falcon_charge_annotation_table.tsv',
    "C24_alkylamine_dihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_dihydroxy/download_mgfs/final_mgf/C24_alkylamine_dihydroxy_input_falcon_charge_annotation_table.tsv',
    "C24_alkylamine_trihydroxy_part1": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part1_annotation_table.tsv',
    "C24_alkylamine_trihydroxy_part2": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part2_annotation_table.tsv',
    "C24_alkylamine_trihydroxy_part3": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part3_annotation_table.tsv',
    "C24_alkylamine_trihydroxy_part4": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part4_annotation_table.tsv',
    "C24_alkylamine_tetrahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_tetrahydroxy/download_mgfs/final_mgf/C24_alkylamine_tetrahydroxy_input_falcon_charge_annotation_table.tsv',
    "C24_alkylamine_pentahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_pentahydroxy/download_mgfs/final_mgf/C24_alkylamine_pentahydroxy_input_falcon_charge_annotation_table.tsv',

    "C28_alcohol_1OH": f'{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/One_OH/download_mgfs/final_mgf/C28_alcohol_1OH_input_falcon_351_C28O1_charge_annotation_table.tsv',
    "C28_alcohol_2OH": f'{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/Two_OH/download_mgfs/final_mgf/C28_alcohol_2OH_input_falcon_367_C28O2_charge_annotation_table.tsv',
    "C28_alcohol_3OH": f'{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/Three_OH/download_mgfs/final_mgf/C28_alcohol_3OH_input_falcon_383_C28O3_charge_annotation_table.tsv',
}

post_falcon = {
    'C24_nonhydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Nonhydroxy/falcon_output/C24_nonhydroxy_falcon_eps01_minsamples2.csv',
    'C24_monohydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Monohydroxy/falcon_output/C24_monohydroxy_falcon_eps01_minsamples2.csv',
    'C24_dihydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Dihydroxy/falcon_output/C24_dihydroxy_falcon_eps01_minsamples2.csv',
    'C24_trihydroxy_part1': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/falcon_output/C24_final_trihydroxy_part1_falcon_eps01_minsamples2.csv',
    'C24_trihydroxy_part2': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/falcon_output/C24_final_trihydroxy_part2_falcon_eps01_minsamples2.csv',
    'C24_trihydroxy_part3': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/falcon_output/C24_final_trihydroxy_part3_falcon_eps01_minsamples2.csv',
    'C24_tetrahydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Tetrahydroxy/falcon_output/C24_tetrahydroxy_falcon_eps01_minsamples2.csv',
    'C24_pentahydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Pentahydroxy/falcon_output/C24_pentahydroxy_falcon_eps01_minsamples2.csv',

    "C23_monohydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_monohydroxy/falcon_output/C23_monohydroxy_falcon_eps01_minsamples2.csv',
    "C23_dihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_dihydroxy/falcon_output/C23_dihydroxy_falcon_eps01_minsamples2.csv',
    "C23_trihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_trihydroxy/falcon_output/C23_trihydroxy_falcon_eps01_minsamples2.csv',
    "C23_tetrahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_tetrahydroxy/falcon_output/C23_tetrahydroxy_falcon_eps01_minsamples2.csv',
    "C23_pentahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_pentahydroxy/falcon_output/C23_pentahydroxy_falcon_eps01_minsamples2.csv',

    "C27_monohydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_monohydroxy/falcon_output/C27_monohydroxy_falcon_eps01_minsamples2.csv',
    "C27_dihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_dihydroxy/falcon_output/C27_dihydroxy_falcon_eps01_minsamples2.csv',
    "C27_trihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_trihydroxy/falcon_output/C27_trihydroxy_falcon_eps01_minsamples2.csv',
    "C27_tetrahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_tetrahydroxy/falcon_output/C27_tetrahydroxy_falcon_eps01_minsamples2.csv',
    "C27_pentahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_pentahydroxy/falcon_output/C27_pentahydroxy_falcon_eps01_minsamples2.csv',

    "Aceto_dihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/Aceto/Dihydroxy/falcon_output/aceto_dihydroxy_falcon_eps01_minsamples2.csv',

    "C24_alkylamine_monohydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_monohydroxy/falcon_output/C24_alkylamine_monohydroxy_falcon_eps01_minsamples2.csv',
    "C24_alkylamine_dihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_dihydroxy/falcon_output/C24_alkylamine_dihydroxy_falcon_eps01_minsamples2.csv',
    "C24_alkylamine_trihydroxy_part1": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/falcon_output/C24_alkylamine_trihydroxy_part1_falcon_eps01_minsamples2.csv',
    "C24_alkylamine_trihydroxy_part2": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/falcon_output/C24_alkylamine_trihydroxy_part2_falcon_eps01_minsamples2.csv',
    "C24_alkylamine_trihydroxy_part3": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/falcon_output/C24_alkylamine_trihydroxy_part3_falcon_eps01_minsamples2.csv',
    "C24_alkylamine_trihydroxy_part4": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/falcon_output/C24_alkylamine_trihydroxy_part4_falcon_eps01_minsamples2.csv',
    "C24_alkylamine_tetrahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_tetrahydroxy/falcon_output/C24_alkylamine_tetrahydroxy_falcon_eps01_minsamples2.csv',
    "C24_alkylamine_pentahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_pentahydroxy/falcon_output/C24_alkylamine_pentahydroxy_falcon_eps01_minsamples2.csv',

    "C28_alcohol_1OH": f'{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/One_OH/falcon_output/C28_alcohol_1OH_falcon_eps01_minsamples2.csv',
    "C28_alcohol_2OH": f'{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/Two_OH/falcon_output/C28_alcohol_2OH_falcon_eps01_minsamples2.csv',
    "C28_alcohol_3OH": f'{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/Three_OH/falcon_output/C28_alcohol_3OH_falcon_eps01_minsamples2.csv',
}

final_mgfs_annotation_table = {
    'C23_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C23_annotation_combined_scan.tsv',
    'C24_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_annotation_combined_scan.tsv',
    'C24_alkamine_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_alkylamine_annotation_combined_scan.tsv',
    'C26_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C26_annotation_combined_scan.tsv',
    'C27_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C27_annotation_combined_scan.tsv',
    'C28_alcohol_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C28_alcohol_annotation_combined_scan.tsv',
}

## Loop

In [ ]:
import pandas as pd
import polars as pl
import os, re
from collections import defaultdict

COMBINED_MASSQL = f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/combined_all_massql_results_cleaned.tsv'
OUTPUT_DIR = f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/BA_core_merged_files_for_plot'
FALCON_SKIPROWS = 25  

def strip_part(key):
    return re.sub(r'_part\d+$', '', key)

def harmonize_token(ba):
    return ba.replace('Aceto', 'C26').replace('alkylamine', 'alkamine')

def get_library_key(ba):
    if ba.startswith('C23'):            return 'C23_library'
    if ba.startswith('C24_alkylamine'): return 'C24_alkamine_library'
    if ba.startswith('C24'):            return 'C24_library'
    if ba.startswith('C27'):            return 'C27_library'
    if ba.startswith('Aceto'):          return 'C26_library'
    if ba.startswith('C28_alcohol'):    return 'C28_alcohol_library'
    raise ValueError(f'no library mapping for {ba}')

def harmonize_library(df):
    df = df.copy()
    df['BA_core'] = df['identifier'].str.split(':').str[2].replace('_input_falcon_charge', '', regex=True)
    for a, b in [('Mono','mono'), ('Non','non'), ('Tri','tri'), ('Aceto','C26'),
                 ('C28_alcohol_3OH_input_falcon_383_C28O3_charge','C28_alcohol_3OH'),
                 ('C28_alcohol_2OH_input_falcon_367_C28O2_charge','C28_alcohol_2OH'),
                 ('C28_alcohol_1OH_input_falcon_351_C28O1_charge','C28_alcohol_1OH'),
                 ('alkylamine','alkamine')]:
        df['BA_core'] = df['BA_core'].replace(a, b, regex=True)
    return df

# group part files under one BA_core
ba_core_keys = defaultdict(list)
for key in pre_falcon:
    ba_core_keys[strip_part(key)].append(key)
for ba in ba_core_keys:
    ba_core_keys[ba].sort()

# load each library once
library_cache = {k: harmonize_library(pd.read_csv(p, sep='\t'))
                 for k, p in final_mgfs_annotation_table.items()}

# load massql once
combined_all = pl.read_csv(COMBINED_MASSQL, separator='\t')

In [ ]:
def pick_pl(df_pl, candidates):
    for c in candidates:
        f = df_pl.filter(pl.col('BA_core') == c)
        if f.height: return f, c
    for c in candidates:
        f = df_pl.filter(pl.col('BA_core').str.starts_with(c))
        if f.height: return f, c
    return df_pl.head(0), None

def pick_pd(df_pd, candidates):
    for c in candidates:
        f = df_pd[df_pd['BA_core'] == c]
        if len(f): return f
    for c in candidates:
        f = df_pd[df_pd['BA_core'].str.contains(c, regex=False, na=False)]
        if len(f): return f
    return df_pd.iloc[0:0]

summary = []
for ba in sorted(ba_core_keys):
    token = harmonize_token(ba)
    candidates = list(dict.fromkeys([token, ba]))   # harmonized first, raw as fallback

    df = pick_pd(library_cache[get_library_key(ba)], candidates)

    comb, _ = pick_pl(combined_all, candidates)
    comb = comb.to_pandas()
    if len(comb):
        comb['original_path'] = comb['original_path'].astype(str).str.strip()
        comb['original_path_scan'] = comb['original_path'] + '_' + comb['scan'].astype(str).str.strip()

    part_frames = []
    for key in ba_core_keys[ba]:
        post = pd.read_csv(post_falcon[key], sep=',', skiprows=FALCON_SKIPROWS)
        post.columns = post.columns.str.strip()
        post['scan_combined_mgf'] = post['identifier'].str.split(':').str[-1].astype(str)
        pre = pd.read_csv(pre_falcon[key], sep='\t')
        pre.columns = pre.columns.str.strip()
        pre['scan_combined_mgf'] = pre['scan_combined_mgf'].astype(str)

        merged = pd.merge(pre, post, on='scan_combined_mgf', how='left')

        keep = pd.merge(df[['identifier']], merged[['identifier', 'cluster']],
                        on='identifier', how='left')['cluster'].dropna().tolist()
        part_frames.append(merged[merged['cluster'].isin(keep)])

    merged_all = pd.concat(part_frames, ignore_index=True)
    merged_all['original_path'] = merged_all['original_path'].astype(str).str.strip()
    merged_all['original_path_scan'] = (
    merged_all['original_path'] + '_' + merged_all['original_scan'].astype(str).str.strip()
)

    if len(comb):
        final_merged = pd.merge(merged_all, comb[['original_path_scan', 'rt']],
                                on='original_path_scan', how='left')
    else:
        final_merged = merged_all.assign(rt=pd.NA)

    out = os.path.join(OUTPUT_DIR, f'{ba}_table_for_artifacts_estimation.tsv')
    final_merged.to_csv(out, sep='\t', index=False)
    summary.append((ba, get_library_key(ba), len(df), len(final_merged),
                    int(final_merged['rt'].isna().sum())))

print(pd.DataFrame(summary, columns=['BA_core','library','lib_rows','out_rows','rt_NaN']).to_string(index=False))

## Determine what is a potential artifact

In [ ]:
#first, let's just check how many 'cluster' unique values we have for each file and how many rows in it each, and how many of those row have rt values

import os
import pandas as pd

folder = f"{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/BA_core_merged_files_for_plot"

def read_table(path):
    # pick separator based on extension (csv = comma, tsv/txt = tab)
    ext = os.path.splitext(path)[1].lower()
    sep = "," if ext == ".csv" else "\t"
    return pd.read_csv(path, sep=sep, low_memory=False)

results = []

for fname in sorted(os.listdir(folder)):
    path = os.path.join(folder, fname)
    if not os.path.isfile(path):
        continue
    try:
        df = read_table(path)
    except Exception as e:
        print(f"Skipping {fname}: {e}")
        continue

    n_rows = len(df)
    n_unique_cluster = df["cluster"].nunique() if "cluster" in df.columns else None
    n_rows_with_rt = df["rt"].notna().sum() if "rt" in df.columns else None

    results.append({
        "file": fname,
        "total_rows": n_rows,
        "unique_clusters": n_unique_cluster,
        "rows_with_rt": n_rows_with_rt,
    })

summary = pd.DataFrame(results)
summary

In [ ]:
# and rename pepmass to precmz in these files

import csv
import pandas as pd
from pathlib import Path

FOLDER = Path(
    f"{PROJECT_DIR}/Scripts_data_analysis/"
    "ISF_evaluation/BA_core_merged_files_for_plot"
)
EXTENSIONS = {".csv", ".tsv", ".txt"}   # set to None to process every file

changed = skipped = 0
for path in sorted(FOLDER.iterdir()):
    if not path.is_file():
        continue
    if EXTENSIONS is not None and path.suffix.lower() not in EXTENSIONS:
        continue

    # sniff the delimiter so .csv and .tsv are both handled and preserved
    with open(path, newline="") as f:
        sep = csv.Sniffer().sniff(f.readline()).delimiter

    df = pd.read_csv(path, sep=sep)

    if "pepmass" not in df.columns:
        skipped += 1
        continue

    df = df.rename(columns={"pepmass": "precmz"})
    df.to_csv(path, sep=sep, index=False)
    print(f"updated: {path.name}")
    changed += 1

print(f"\nDone. {changed} file(s) updated, {skipped} skipped.")

### Per core

In [ ]:
#calculate potential artifacts and create Sankey diagrams

import os, re, importlib
import pandas as pd
import plotly.graph_objects as go

import artifact_screen
importlib.reload(artifact_screen)
from artifact_screen import screen_artifacts, ISOTOPE_ADDUCT_OFFSETS, ISF_NEUTRAL_LOSS_OFFSETS

in_folder  = f"{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/BA_core_merged_files_for_plot"
out_folder = f"{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts"
os.makedirs(out_folder, exist_ok=True)

# ---------------------------------------------------------------------------
# Classification + Sankey code
# ---------------------------------------------------------------------------
CLASS_PRIORITY = ["Isotope", "Adduct", "In-source fragment"]

COL = {
    "root":     "#C9CDD4",
    "genuine":  "#8E1B5B",
    "artifact": "#E0A800",
    "Isotope":  "#E8B92E",
    "Adduct":   "#D98A1F",
    "In-source fragment": "#C9A227",
}
SPECIES_COL = "#EBC547"

def classify(type_str):
    parts = [p.strip() for p in str(type_str).split(";") if p.strip()]
    found = []
    for p in parts:
        pl = p.lower()
        if "isotope" in pl:
            base = "M+1" if "m+1" in pl else ("M+2" if "m+2" in pl else "isotope")
            sp = base + (" (2+)" if "2+" in pl else "")
            found.append(("Isotope", sp))
        elif "adduct" in pl:
            sp = ("+Na"  if pl.startswith("na")  else
                  "+NH4" if pl.startswith("nh4") else
                  "+K"   if pl.startswith("k")   else
                  "+Ca"  if pl.startswith("ca")  else "+?")
            found.append(("Adduct", sp))
        elif pl.startswith("isf"):
            m = re.search(r"-\s*(.+)$", p)
            loss = (m.group(1).strip() if m else p)
            loss = loss.replace("H2O", "H\u2082O").replace("CO2", "CO\u2082").replace("trimethylamine", "TMA")
            found.append(("In-source fragment", "\u2212" + loss))
    if not found:
        return None
    rank = {c: i for i, c in enumerate(CLASS_PRIORITY)}
    found.sort(key=lambda cs: rank.get(cs[0], 99))
    return found[0]

def build_sankey(annotated, title=None, height=560, width=1000, pct_base="total",
                 show_pct=True, pct_decimals=1, text_color="#DADADA"):
    n_total = len(annotated)
    cls, spec = [], []
    for _, r in annotated.iterrows():
        if not r["is_artifact"]:
            cls.append("Genuine analog"); spec.append(None); continue
        c = classify(r["artifact_types"])
        if c is None:
            cls.append("Genuine analog"); spec.append(None); continue
        cls.append(c[0]); spec.append(c[1])
    a = annotated.assign(_class=cls, _species=spec)

    n_gen = int((a["_class"] == "Genuine analog").sum())
    n_art = n_total - n_gen

    def pct(v, base):
        d = n_total if base == "total" else max(n_art, 1)
        return f"{100 * v / d:.{pct_decimals}f}%" if d else "0%"
    def lab(name, v, base="total"):
        if show_pct:
            return f"{name}  {v} ({pct(v, base)})"
        return f"{name}  {v}"

    labels, colors, node_x, idx = [], [], [], {}
    def node(name, color, x, disp):
        if name not in idx:
            idx[name] = len(labels)
            labels.append(disp); colors.append(color); node_x.append(x)
        return idx[name]

    s, t, val, lcol, lpct = [], [], [], [], []
    def al(h, af=0.55):
        h = h.lstrip("#")
        return f"rgba({int(h[0:2],16)},{int(h[2:4],16)},{int(h[4:6],16)},{af})"
    def link(x, y, v, c):
        if v > 0:
            s.append(x); t.append(y); val.append(v); lcol.append(c)
            lpct.append(100 * v / n_total if n_total else 0)

    root = node("Library entries", COL["root"], 0.01, lab("Library entries", n_total))
    gen  = node("Analog",  COL["genuine"], 0.30, lab("Analog", n_gen))
    art  = node("Artifact",        COL["artifact"], 0.30, lab("Other ion forms", n_art))   # CHANGED label
    link(root, gen, n_gen, al(COL["genuine"]))
    link(root, art, n_art, al(COL["artifact"]))

    kept = node("Genuine analog (kept)", COL["genuine"], 0.97, lab("M+H", n_gen))           # CHANGED label
    link(gen, kept, n_gen, al(COL["genuine"], 0.45))

    adf = a[a["_class"] != "Genuine analog"]
    for k in ["Isotope", "Adduct", "In-source fragment"]:
        sub = adf[adf["_class"] == k]
        if sub.empty:
            continue
        kn = node(k, COL[k], 0.58, lab(k, len(sub), pct_base))
        link(art, kn, len(sub), al(COL[k]))
        for sp, ss in sub.groupby("_species"):
            spn = node(f"{k}:{sp}", SPECIES_COL, 0.97, lab(sp, len(ss), pct_base))
            link(kn, spn, len(ss), al(SPECIES_COL, 0.5))

    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(label=labels, color=colors, x=node_x, pad=18, thickness=20,
                  line=dict(width=0), hovertemplate="%{label}<extra></extra>"),
        link=dict(source=s, target=t, value=val, color=lcol, customdata=lpct,
                  hovertemplate="%{value} entries (%{customdata:.1f}% of all)<extra></extra>"),
    ))
    fig.update_layout(
        title=dict(text=title or "", x=0.5, xanchor="center",                                # ADDED title
                   font=dict(family="Inter, Arial", size=16, color="#333333")),
        font=dict(family="Inter, Arial", size=13, color=text_color),
        paper_bgcolor="white", plot_bgcolor="white",
        margin=dict(l=10, r=10, t=70, b=20), height=height, width=width,
    )
    return fig

# ---------------------------------------------------------------------------
# Loop over every file in the input folder
# ---------------------------------------------------------------------------
def read_table(path):
    ext = os.path.splitext(path)[1].lower()
    sep = "," if ext == ".csv" else "\t"
    return pd.read_csv(path, sep=sep, low_memory=False)

for fname in sorted(os.listdir(in_folder)):
    path = os.path.join(in_folder, fname)
    if not os.path.isfile(path):
        continue
    base = os.path.splitext(fname)[0]
    try:
        combined_rt = read_table(path)

        pairs, annotated = screen_artifacts(
            combined_rt,
            offsets=ISOTOPE_ADDUCT_OFFSETS + ISF_NEUTRAL_LOSS_OFFSETS,
            ppm_tol=20.0, rt_tol_sec=10.0,
        )

        annotated.to_csv(os.path.join(out_folder, f"{base}_annotated_artifacts.tsv"),
                         sep="\t", index=False)
        pairs.to_csv(os.path.join(out_folder, f"{base}_pairs_artifact.tsv"),
                     sep="\t", index=False)

        fig = build_sankey(annotated, title=base)                                            # title = file name
        fig.write_html(os.path.join(out_folder, f"{base}_sankey.html"))

        # try:                                                                                 # ADDED pdf export
        #     fig.write_image(os.path.join(out_folder, f"{base}_sankey.pdf"))
        # except Exception as e:
        #     print(f"     (PDF export skipped for {fname}: {e})")

        print(f"OK   {fname}: {len(annotated)} entries, {len(pairs)} pairs")
    except Exception as e:
        print(f"FAIL {fname}: {e}")

### Cores combined

In [ ]:
import os, re, glob
from collections import defaultdict

# ---- settings ----
INPUT_FOLDER  = f"{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/BA_core_merged_files_for_plot"              # folder containing the TSVs
OUTPUT_FOLDER = f"{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/BA_core_merged_files_for_plot/cores_combined_6_total"     # where the *_combined.tsv files get written
HAS_HEADER    = True             # True: keep header from 1st file only; False: keep every line
SOURCE_COL    = "source_file"    # name of the added column
# ------------------

SUFFIX = "_table_for_artifacts_estimation.tsv"
DESCRIPTOR = re.compile(r"_(?:[A-Za-z]*hydroxy|\d+OH)$", re.IGNORECASE)

def prefix_of(name):
    stem = name[:-len(SUFFIX)]
    m = DESCRIPTOR.search(stem)
    return stem[:m.start()] if m else stem

def add_col(line, value):
    """append a tab-separated column to one line, preserving the newline"""
    return line.rstrip("\n") + "\t" + value + "\n"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

groups = defaultdict(list)
for path in glob.glob(os.path.join(INPUT_FOLDER, "*" + SUFFIX)):
    name = os.path.basename(path)
    if name.endswith("_combined.tsv"):       # don't re-merge our own output
        continue
    groups[prefix_of(name)].append(path)

if not groups:
    print(f"No files matching *{SUFFIX} found in {os.path.abspath(INPUT_FOLDER)}")

for prefix, files in sorted(groups.items()):
    files.sort()
    out_path = os.path.join(OUTPUT_FOLDER, f"{prefix}_combined.tsv")
    with open(out_path, "w", newline="") as out:
        for i, f in enumerate(files):
            src = os.path.basename(f)
            with open(f, newline="") as fin:
                lines = fin.readlines()
            if not lines:
                continue
            for j, line in enumerate(lines):
                if HAS_HEADER and j == 0:        # header row of this file
                    if i == 0:                   # write header only for first file
                        out.write(add_col(line, SOURCE_COL))
                    continue                     # skip later files' headers
                out.write(add_col(line, src))    # data row -> tag with filename
    print(f"{prefix:<18} <- {len(files):>2} files  ->  {out_path}")

In [ ]:
#calculate potential artifacts and create Sankey diagrams

import os, re, importlib
import pandas as pd
import plotly.graph_objects as go

import artifact_screen
importlib.reload(artifact_screen)
from artifact_screen import screen_artifacts, ISOTOPE_ADDUCT_OFFSETS, ISF_NEUTRAL_LOSS_OFFSETS

in_folder  = f"{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/BA_core_merged_files_for_plot/cores_combined_6_total"
out_folder = f"{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/cores_combined_6_total"
os.makedirs(out_folder, exist_ok=True)

# ---------------------------------------------------------------------------
# Classification + Sankey code
# ---------------------------------------------------------------------------
CLASS_PRIORITY = ["Isotope", "Adduct", "In-source fragment"]

COL = {
    "root":     "#C9CDD4",
    "genuine":  "#8E1B5B",
    "artifact": "#E0A800",
    "Isotope":  "#E8B92E",
    "Adduct":   "#D98A1F",
    "In-source fragment": "#C9A227",
}
SPECIES_COL = "#EBC547"

def classify(type_str):
    parts = [p.strip() for p in str(type_str).split(";") if p.strip()]
    found = []
    for p in parts:
        pl = p.lower()
        if "isotope" in pl:
            base = "M+1" if "m+1" in pl else ("M+2" if "m+2" in pl else "isotope")
            sp = base + (" (2+)" if "2+" in pl else "")
            found.append(("Isotope", sp))
        elif "adduct" in pl:
            sp = ("+Na"  if pl.startswith("na")  else
                  "+NH4" if pl.startswith("nh4") else
                  "+K"   if pl.startswith("k")   else
                  "+Ca"  if pl.startswith("ca")  else "+?")
            found.append(("Adduct", sp))
        elif pl.startswith("isf"):
            m = re.search(r"-\s*(.+)$", p)
            loss = (m.group(1).strip() if m else p)
            loss = loss.replace("H2O", "H\u2082O").replace("CO2", "CO\u2082").replace("trimethylamine", "TMA")
            found.append(("In-source fragment", "\u2212" + loss))
    if not found:
        return None
    rank = {c: i for i, c in enumerate(CLASS_PRIORITY)}
    found.sort(key=lambda cs: rank.get(cs[0], 99))
    return found[0]

def build_sankey(annotated, title=None, height=560, width=1000, pct_base="total",
                 show_pct=True, pct_decimals=1, text_color="#DADADA"):
    n_total = len(annotated)
    cls, spec = [], []
    for _, r in annotated.iterrows():
        if not r["is_artifact"]:
            cls.append("Genuine analog"); spec.append(None); continue
        c = classify(r["artifact_types"])
        if c is None:
            cls.append("Genuine analog"); spec.append(None); continue
        cls.append(c[0]); spec.append(c[1])
    a = annotated.assign(_class=cls, _species=spec)

    n_gen = int((a["_class"] == "Genuine analog").sum())
    n_art = n_total - n_gen

    def pct(v, base):
        d = n_total if base == "total" else max(n_art, 1)
        return f"{100 * v / d:.{pct_decimals}f}%" if d else "0%"
    def lab(name, v, base="total"):
        if show_pct:
            return f"{name}  {v} ({pct(v, base)})"
        return f"{name}  {v}"

    labels, colors, node_x, idx = [], [], [], {}
    def node(name, color, x, disp):
        if name not in idx:
            idx[name] = len(labels)
            labels.append(disp); colors.append(color); node_x.append(x)
        return idx[name]

    s, t, val, lcol, lpct = [], [], [], [], []
    def al(h, af=0.55):
        h = h.lstrip("#")
        return f"rgba({int(h[0:2],16)},{int(h[2:4],16)},{int(h[4:6],16)},{af})"
    def link(x, y, v, c):
        if v > 0:
            s.append(x); t.append(y); val.append(v); lcol.append(c)
            lpct.append(100 * v / n_total if n_total else 0)

    root = node("Library entries", COL["root"], 0.01, lab("Library entries", n_total))
    gen  = node("Analog",  COL["genuine"], 0.30, lab("Analog", n_gen))
    art  = node("Artifact",        COL["artifact"], 0.30, lab("Other ion forms", n_art))   # CHANGED label
    link(root, gen, n_gen, al(COL["genuine"]))
    link(root, art, n_art, al(COL["artifact"]))

    kept = node("Genuine analog (kept)", COL["genuine"], 0.97, lab("M+H", n_gen))           # CHANGED label
    link(gen, kept, n_gen, al(COL["genuine"], 0.45))

    adf = a[a["_class"] != "Genuine analog"]
    for k in ["Isotope", "Adduct", "In-source fragment"]:
        sub = adf[adf["_class"] == k]
        if sub.empty:
            continue
        kn = node(k, COL[k], 0.58, lab(k, len(sub), pct_base))
        link(art, kn, len(sub), al(COL[k]))
        for sp, ss in sub.groupby("_species"):
            spn = node(f"{k}:{sp}", SPECIES_COL, 0.97, lab(sp, len(ss), pct_base))
            link(kn, spn, len(ss), al(SPECIES_COL, 0.5))

    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(label=labels, color=colors, x=node_x, pad=18, thickness=20,
                  line=dict(width=0), hovertemplate="%{label}<extra></extra>"),
        link=dict(source=s, target=t, value=val, color=lcol, customdata=lpct,
                  hovertemplate="%{value} entries (%{customdata:.1f}% of all)<extra></extra>"),
    ))
    fig.update_layout(
        title=dict(text=title or "", x=0.5, xanchor="center",                                # ADDED title
                   font=dict(family="Inter, Arial", size=16, color="#333333")),
        font=dict(family="Inter, Arial", size=13, color=text_color),
        paper_bgcolor="white", plot_bgcolor="white",
        margin=dict(l=10, r=10, t=70, b=20), height=height, width=width,
    )
    return fig

# ---------------------------------------------------------------------------
# Loop over every file in the input folder
# ---------------------------------------------------------------------------
def read_table(path):
    ext = os.path.splitext(path)[1].lower()
    sep = "," if ext == ".csv" else "\t"
    return pd.read_csv(path, sep=sep, low_memory=False)

for fname in sorted(os.listdir(in_folder)):
    path = os.path.join(in_folder, fname)
    if not os.path.isfile(path):
        continue
    base = os.path.splitext(fname)[0]
    try:
        combined_rt = read_table(path)

        pairs, annotated = screen_artifacts(
            combined_rt,
            offsets=ISOTOPE_ADDUCT_OFFSETS + ISF_NEUTRAL_LOSS_OFFSETS,
            ppm_tol=20.0, rt_tol_sec=10.0,
        )

        annotated.to_csv(os.path.join(out_folder, f"{base}_annotated_artifacts.tsv"),
                         sep="\t", index=False)
        pairs.to_csv(os.path.join(out_folder, f"{base}_pairs_artifact.tsv"),
                     sep="\t", index=False)

        fig = build_sankey(annotated, title=base)                                            # title = file name
        fig.write_html(os.path.join(out_folder, f"{base}_sankey.html"))

        # try:                                                                                 # ADDED pdf export
        #     fig.write_image(os.path.join(out_folder, f"{base}_sankey.pdf"))
        # except Exception as e:
        #     print(f"     (PDF export skipped for {fname}: {e})")

        print(f"OK   {fname}: {len(annotated)} entries, {len(pairs)} pairs")
    except Exception as e:
        print(f"FAIL {fname}: {e}")

# Update the compound names in the libraries

In [ ]:
artifacts = {
    'C24_nonhydroxy': f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C24_nonhydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv', 
    'C24_monohydroxy': f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C24_monohydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    'C24_dihydroxy': f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C24_dihydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    'C24_trihydroxy': f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C24_trihydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    'C24_tetrahydroxy': f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C24_tetrahydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    'C24_pentahydroxy': f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C24_pentahydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',

    "C23_monohydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C23_monohydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C23_dihydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C23_dihydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C23_trihydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C23_trihydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C23_tetrahydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C23_tetrahydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C23_pentahydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C23_pentahydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',

    "C27_monohydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C27_monohydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C27_dihydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C27_dihydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C27_trihydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C27_trihydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C27_tetrahydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C27_tetrahydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C27_pentahydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C27_pentahydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',

    "C26_dihydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/Aceto_dihydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',

    "C24_alkylamine_monohydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C24_alkylamine_monohydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C24_alkylamine_dihydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C24_alkylamine_dihydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C24_alkylamine_trihydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C24_alkylamine_trihydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C24_alkylamine_tetrahydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C24_alkylamine_tetrahydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C24_alkylamine_pentahydroxy": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C24_alkylamine_pentahydroxy_table_for_artifacts_estimation_annotated_artifacts.tsv',

    "C28_alcohol_1OH": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C28_alcohol_1OH_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C28_alcohol_2OH": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C28_alcohol_2OH_table_for_artifacts_estimation_annotated_artifacts.tsv',
    "C28_alcohol_3OH": f'{PROJECT_DIR}/Scripts_data_analysis/ISF_evaluation/TSV_and_HTML_files_artifacts/C28_alcohol_3OH_table_for_artifacts_estimation_annotated_artifacts.tsv'
}

annotation_table = {
    'C23_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C23_annotation_combined_scan.tsv',
    'C24_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_annotation_combined_scan.tsv',
    'C24_alkamine_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_alkylamine_annotation_combined_scan.tsv',
    'C26_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C26_annotation_combined_scan.tsv',
    'C27_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C27_annotation_combined_scan.tsv',
    'C28_alcohol_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C28_alcohol_annotation_combined_scan.tsv',
}

final_annotation_table = {
    'C23_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C23_library_final_annotation.tsv',
    'C24_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_library_final_annotation.tsv',
    'C24_alkamine_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_alkylamine_library_final_annotation.tsv',
    'C26_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C26_library_final_annotation.tsv',
    'C27_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C27_library_final_annotation.tsv',
    'C28_alcohol_library': f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C28_alcohol_library_final_annotation.tsv'
}

In [ ]:
library_to_artifacts = {
    'C23_library':          ['C23_monohydroxy', 'C23_dihydroxy', 'C23_trihydroxy',
                             'C23_tetrahydroxy', 'C23_pentahydroxy'],
    'C24_library':          ['C24_nonhydroxy', 'C24_monohydroxy', 'C24_dihydroxy',
                             'C24_trihydroxy', 'C24_tetrahydroxy', 'C24_pentahydroxy'],
    'C24_alkamine_library': ['C24_alkylamine_monohydroxy', 'C24_alkylamine_dihydroxy',
                             'C24_alkylamine_trihydroxy', 'C24_alkylamine_tetrahydroxy',
                             'C24_alkylamine_pentahydroxy'],
    'C26_library':          ['C26_dihydroxy'],
    'C27_library':          ['C27_monohydroxy', 'C27_dihydroxy', 'C27_trihydroxy',
                             'C27_tetrahydroxy', 'C27_pentahydroxy'],
    'C28_alcohol_library':  ['C28_alcohol_1OH', 'C28_alcohol_2OH', 'C28_alcohol_3OH'],
}

In [ ]:
import pandas as pd

def process_library(lib_key, artifact_keys):
    # 1. Combine artifact tables for this library and flag artifact clusters.
    #    is_artifact_cluster is computed *within each file*, because 'cluster'
    #    IDs may repeat across files.
    parts = []
    for ak in artifact_keys:
        df = pd.read_csv(artifacts[ak], sep='\t')
        df['is_artifact_cluster'] = df.groupby('cluster')['is_artifact'].transform('any')
        parts.append(df[['identifier', 'is_artifact_cluster']])

    artifact_flags = pd.concat(parts, ignore_index=True)
    # collapse any duplicate identifiers: flagged if ANY file flags it
    artifact_flags = artifact_flags.groupby('identifier', as_index=False)['is_artifact_cluster'].any()

    # 2. Merge onto the combined-scan annotation table
    annotation = pd.read_csv(annotation_table[lib_key], sep='\t')
    merged = pd.merge(annotation, artifact_flags, on='identifier', how='left')
    merged = merged.drop_duplicates()

    # 3. Bring the flag onto the final annotation table
    annotation_final = pd.read_csv(final_annotation_table[lib_key], sep='\t')
    annotation_final = annotation_final.drop(columns=['Unnamed: 0'], errors='ignore')

    merged = merged.rename(columns={'scan_final_mgf': 'EXTRACTSCAN'})
    merged_final = pd.merge(
        annotation_final,
        merged[['EXTRACTSCAN', 'is_artifact_cluster']],
        on='EXTRACTSCAN', how='left'
    )

    # 4. delta_mass from COMPOUND_NAME
    merged_final['delta_mass'] = (
        merged_final['COMPOUND_NAME']
        .str.extract(r'delta mass\s*(-?\d+\.?\d*)')
        .astype(float)
    )

    # 5. Flag positive-delta artifact clusters in COMPOUND_NAME
    mask = (merged_final['is_artifact_cluster'] == True) & (merged_final['delta_mass'] > 0)
    merged_final.loc[mask, 'COMPOUND_NAME'] = (
        merged_final.loc[mask, 'COMPOUND_NAME'].astype(str) + "; potential ISF/adduct/isotope"
    )

    # 6. Fix file extension
    merged_final['FILENAME'] = merged_final['FILENAME'].str.replace('.mgf$', '.mzXML', regex=True)

    # 7. Drop columns that are not accepted
    merged_final = merged_final.drop(columns=['delta_mass', 'is_artifact_cluster'], errors='ignore')

    # 8. Subtract 1.0073
    merged_final['EXACTMASS'] = merged_final['MOLECULEMASS'] - 1.0073

    # 9. Export
    out_path = final_annotation_table[lib_key].replace('.tsv', '_artifacts.tsv')
    merged_final.to_csv(out_path, sep='\t', index=False)
    return out_path


for lib_key, artifact_keys in library_to_artifacts.items():
    out_path = process_library(lib_key, artifact_keys)
    print(f'{lib_key}: wrote {out_path}')